In [1]:
import sys
from pathlib import Path

sys.path.append(f"{Path().absolute().parent}")

In [2]:
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gpytorch")

In [3]:
from apps.mobility_robustness_optimization.mobility_robustness_optimization import *
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO

In [4]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [5]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

## Simple MRO

In [6]:
mro = SimpleMRO(params, topology)

In [7]:
mro.update(ue_data)

No Bayesian Digital Twins available for update. Training from scratch.


[2025-04-30 20:25:36,617] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-04-30 20:25:36,671] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019082)
[2025-04-30 20:25:36,724] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019101)
[2025-04-30 20:25:36,772] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019149)
[2025-04-30 20:25:36,821] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019220)
[2025-04-30 20:25:36,863] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019362)
[2025-04-30 20:25:36,902] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019526)
[2025-04-30 20:25:36,938] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019712)
[2025-04-30 20:25:37,009] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019937)
[2025-04-30 20:25:37,055] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020143)
[2025-04-30 20:25:37,106] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020333)
[2025-04-30 20:25:37,160] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020556)
[2025-04-30 20:25:37,207] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020724)
[2025-04-30 20

In [8]:
hyst,ttt = mro.solve()

/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gpytorch/distributions/multivariate_normal.py:319

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      2.2184735496   26     99.650000   
1      1.4393060732   97     99.150000   
2      2.0128746796   7      100.000000  
3      3.8402074362   63     99.350000   
4      1.1621166629   78     99.300000   
5      0.1888093189   77     99.300000   
6      3.5983932038   90     99.150000   
7      1.9375820914   78     99.300000   
8      3.2105858979   66     99.350000   
9      0.8197289450   60     99.350000   
10     2.3509957924   42     99.550000   
11     3.7939120644   77     99.300000   
12     4.2171221311   20     99.650000   
13     2.8001001152   39     99.550000   
14     3.4531762070   34     99.550000   
15     0.3127531349   13     99.950000   
16     1.1625219014   27     99.650000   
17     3.3871334289   9      100.000000  
18     3.3527932357   49     99.450000   
19     0.2619077548   37     99.550000   
20     1.6907413759   90     99.150000   
21     1.6846573505   77     99.30

## RL MRO

In [9]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

In [10]:
mro = ReinforcedMRO(params, topology)

In [11]:
mro.update(ue_data)

[2025-04-30 20:28:45,089] INFO:  Iter 1/100 - Loss: 0.759 (delta=inf)


No Bayesian Digital Twins available for update. Training from scratch.


[2025-04-30 20:28:45,123] INFO:  Iter 2/100 - Loss: 0.741 (delta=-0.018538)
[2025-04-30 20:28:45,155] INFO:  Iter 3/100 - Loss: 0.722 (delta=-0.018627)
[2025-04-30 20:28:45,187] INFO:  Iter 4/100 - Loss: 0.703 (delta=-0.018769)
[2025-04-30 20:28:45,218] INFO:  Iter 5/100 - Loss: 0.684 (delta=-0.018916)
[2025-04-30 20:28:45,250] INFO:  Iter 6/100 - Loss: 0.665 (delta=-0.019115)
[2025-04-30 20:28:45,281] INFO:  Iter 7/100 - Loss: 0.646 (delta=-0.019346)
[2025-04-30 20:28:45,312] INFO:  Iter 8/100 - Loss: 0.626 (delta=-0.019512)
[2025-04-30 20:28:45,343] INFO:  Iter 9/100 - Loss: 0.607 (delta=-0.019731)
[2025-04-30 20:28:45,374] INFO:  Iter 10/100 - Loss: 0.587 (delta=-0.019914)
[2025-04-30 20:28:45,405] INFO:  Iter 11/100 - Loss: 0.567 (delta=-0.020111)
[2025-04-30 20:28:45,436] INFO:  Iter 12/100 - Loss: 0.546 (delta=-0.020305)
[2025-04-30 20:28:45,467] INFO:  Iter 13/100 - Loss: 0.526 (delta=-0.020492)
[2025-04-30 20:28:45,498] INFO:  Iter 14/100 - Loss: 0.505 (delta=-0.020683)
[2025-0

In [12]:
hyst,ttt = mro.solve()

/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/Users/watermenon/Desktop/Repositories/maveric/.venv/lib/python3.9/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Using cpu device
Episode: 1, Timestep: 1, Hyst: 0.000000, TTT: 2, Reward: 88.700000, Done: False
Episode: 1, Timestep: 2, Hyst: 0.756156, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 3, Hyst: 0.196881, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 4, Hyst: 0.935034, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 5, Hyst: 0.000000, TTT: 2, Reward: 88.700000, Done: False
Episode: 1, Timestep: 6, Hyst: 0.000000, TTT: 2, Reward: 88.700000, Done: False
Episode: 1, Timestep: 7, Hyst: 0.937791, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 8, Hyst: 0.000000, TTT: 2, Reward: 88.700000, Done: False
Episode: 1, Timestep: 9, Hyst: 1.138527, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 10, Hyst: 0.128241, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 11, Hyst: 0.814089, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep: 12, Hyst: 0.670747, TTT: 2, Reward: 90.900000, Done: False
Episode: 1, Timestep